# Поиск эталонной проводки: зачисление эквайринга Сбера

Источник: платёжное поручение от коллеги.

- **Плательщик:** Северо-Западный банк ПАО Сбербанк (ИНН `7707083893`)
- **Получатель (клиент РСХБ):** ООО «КАЛА Я МАРЬЯПОЯТ» (ИНН `1017000071`), сч. `40702810035530000002`
- **Ожидаемое назначение:** `ЗАЧИСЛЕНИЕ СРЕДСТВ ПО ОПЕРАЦИЯМ ЭКВАЙРИНГА. МЕРЧАНТ №251000008495. ...`
- **Дата / сумма:** ~21–22.06.2026, `68932.64`

Работаем **только** с `ods.scd1_z_main_docum` (без джойнов к другим таблицам).

In [ ]:
import re
import time

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_rows', 200)

print('Imports loaded')

In [ ]:
# === Эталон с фото ===
table_name = 'ods.scd1_z_main_docum'

client_account = '40702810035530000002'
client_inn = '1017000071'
payer_account = '30233810825000117000'
payer_inn_sber = '7707083893'
merchant_id = '251000008495'
amount_expected = 68932.64

nazn_sample = (
    'ЗАЧИСЛЕНИЕ СРЕДСТВ ПО ОПЕРАЦИЯМ ЭКВАЙРИНГА. '
    'МЕРЧАНТ №251000008495. '
    'КОМИССИЯ 1 372.36 (В Т.Ч. НДС 247.47). '
    'ВОЗВРАТ ПОКУПКИ 0.00/0.00.'
)

# Весь июнь 2026
date_from = '2026-06-01'
date_to_exclusive = '2026-07-01'
date_month_like = '2026-06%'  # для varchar-дат в ISO-подобном виде

# Типы колонок (из ODS)
# c_num_kt/dt  varchar  — номер счёта (основной поиск)
# c_acc_kt/dt  decimal  — id счёта, не 20-значный номер
# c_date_*     varchar
# c_nazn       varchar
# c_sum        decimal  — сумма Дт
# c_sum_po     decimal  — сумма Кт
column_types_fixed = {
    'c_num_kt': 'varchar',
    'c_num_dt': 'varchar',
    'c_acc_kt': 'decimal',
    'c_acc_dt': 'decimal',
    'c_date_prov': 'varchar',
    'c_date_inp': 'varchar',
    'c_date_rec': 'varchar',
    'c_nazn': 'varchar',
    'c_sum': 'decimal',
    'c_sum_po': 'decimal',
}

# Impala
impala_db = 'sandbox_ai'
impala_queue = 'ai'
impala_user_name = 'Shestopalov-VYur'
impala_keytab_path = '/home/jovyan/test_requests/tech.keytab'
impala_use_credentials = True
impala_update_keytab = True
mem_limit = '32g'
preview_limit = 50

# Сначала ищем номер счёта в varchar-полях
account_num_cols = ['c_num_kt', 'c_num_dt']
# decimal id — отдельно, только если нужно (полный 20-знак. номер туда может не влезать)
account_id_cols = ['c_acc_kt', 'c_acc_dt']
date_cols = ['c_date_prov', 'c_date_inp', 'c_date_rec']

out_hit_path = './sber_acq_credit_gold_hit.csv'
out_client_hit_path = './sber_acq_client_account_hit.csv'
out_similar_nazn_path = './sber_acq_credit_similar_nazn.csv'

print(f'table={table_name}')
print(f'period june via dates {date_cols}, like={date_month_like}')
print(f'mem_limit={mem_limit}')
print(f'client_account={client_account} -> search in {account_num_cols}')
print('types:', column_types_fixed)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': impala_queue},
    kerberos={
        'keytab_path': impala_keytab_path,
        'use_credentials': impala_use_credentials,
        'update_keytab': impala_update_keytab,
    },
    user_params={'user_name': impala_user_name},
)
imp._init_connection()
print('Impala connection initialized')

## 1) Схема таблицы — какие колонки есть для счёта / ИНН / суммы / назначения

In [ ]:
def pick_first_existing(columns, candidates):
    colset = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in colset:
            return colset[cand.lower()]
    return None


def find_columns_by_substrings(columns, substrings):
    found = []
    for col in columns:
        low = str(col).lower()
        if any(s in low for s in substrings):
            found.append(col)
    return found


def sql_digits_only(expr):
    return f"regexp_replace(trim(cast({expr} as string)), '[^0-9]', '')"


t0 = time.perf_counter()
with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    schema_raw_df = imp.fetch(f'DESCRIBE {table_name}')
print(f'DESCRIBE done in {round(time.perf_counter() - t0, 2)}s')

name_col = pick_first_existing(schema_raw_df.columns, ['name', 'col_name', 'column_name', 'field'])
type_col = pick_first_existing(schema_raw_df.columns, ['type', 'data_type', 'col_type'])
if name_col is None:
    name_col = schema_raw_df.columns[0]

table_columns = [
    str(x).strip()
    for x in schema_raw_df[name_col].tolist()
    if str(x).strip() and not str(x).startswith('#')
]

# name -> type (lowercase), чтобы правильно писать литералы в WHERE
column_types = {}
if type_col is not None:
    for _, row in schema_raw_df.iterrows():
        cname = str(row[name_col]).strip()
        ctype = str(row[type_col]).strip().lower()
        if cname and not cname.startswith('#'):
            column_types[cname] = ctype
print('account column types:')
for c in ['c_num_kt', 'c_num_dt', 'c_acc_kt', 'c_acc_dt']:
    print(f'  {c}: {column_types.get(c, column_types.get(c.lower(), "?"))}')

field_map = {
    'c_nazn': pick_first_existing(table_columns, ['c_nazn', 'nazn']),
    'c_date_prov': pick_first_existing(table_columns, ['c_date_prov', 'date_prov', 'c_date']),
    'amount': pick_first_existing(table_columns, ['c_sum', 'c_summa', 'summa', 'amount', 'c_sum_nat']),
    'acc_dt': pick_first_existing(table_columns, ['c_acc_dt', 'acc_dt', 'c_acc_a']),
    'acc_kt': pick_first_existing(table_columns, ['c_acc_kt', 'acc_kt', 'c_acc_b']),
    'num_dt': pick_first_existing(table_columns, ['c_num_dt', 'num_dt']),
    'num_kt': pick_first_existing(table_columns, ['c_num_kt', 'num_kt']),
    'inn_kt': pick_first_existing(table_columns, ['c_kl_kt_2_inn', 'c_inn_kt']),
    'inn_dt': pick_first_existing(table_columns, ['c_kl_dt_2_inn', 'c_inn_dt']),
}

resolved_num_cols = [c for c in account_num_cols if pick_first_existing(table_columns, [c])]
resolved_date_cols = [c for c in date_cols if pick_first_existing(table_columns, [c])]
resolved_account_cols = resolved_num_cols  # для summary: ищем именно номер

amount_cols = find_columns_by_substrings(table_columns, ['summa', 'c_sum', 'amount'])

field_map_df = pd.DataFrame([{'role': k, 'column': v} for k, v in field_map.items()])
display(field_map_df)
print(f'resolved_num_cols: {resolved_num_cols}')
print(f'resolved_date_cols: {resolved_date_cols}')
print(f'total columns: {len(table_columns)}')

nazn_col = field_map['c_nazn']
date_col = field_map['c_date_prov']
amount_col = field_map['amount'] or (amount_cols[0] if amount_cols else None)
acc_dt_col = field_map['acc_dt']
acc_kt_col = field_map['acc_kt']
num_dt_col = field_map['num_dt']
num_kt_col = field_map['num_kt']

if nazn_col is None:
    raise RuntimeError('c_nazn not found in schema')
if not resolved_num_cols:
    print('WARNING: c_num_kt/c_num_dt не найдены')
if not resolved_date_cols:
    print('WARNING: колонки дат не найдены')

## 2) Как найти счёт (весь июнь)

**Сначала** ищем номер счёта в varchar:
- `c_num_kt = '40702810035530000002'`
- `c_num_dt = '40702810035530000002'`

`c_acc_kt` / `c_acc_dt` — decimal id, не текстовый 20-значный номер; в первый проход не используем.

Период июня по **трём** varchar-датам (OR):
`c_date_prov`, `c_date_inp`, `c_date_rec` — условие `like '2026-06%'`.

In [ ]:
# Поиск счёта: varchar c_num_* + июнь по трём varchar-датам.

# Подтверждаем наличие колонок (из DESCRIBE / фиксированного списка)
num_cols_ok = [c for c in account_num_cols if pick_first_existing(table_columns, [c])]
date_cols_ok = [c for c in date_cols if pick_first_existing(table_columns, [c])]
if not num_cols_ok:
    raise RuntimeError('Нет c_num_kt/c_num_dt в схеме — искать номер счёта негде')
if not date_cols_ok:
    raise RuntimeError('Нет колонок дат c_date_prov/inp/rec')

select_parts = [
    'c_nazn',
    'c_date_prov', 'c_date_inp', 'c_date_rec',
    'c_sum', 'c_sum_po',
    'c_num_kt', 'c_num_dt',
    'c_acc_kt', 'c_acc_dt',
]
select_parts = [c for c in select_parts if pick_first_existing(table_columns, [c])]
select_sql = ',\n    '.join(select_parts)

# Счёт: только varchar-номер (с кавычками)
account_where = ' OR '.join(f"{col} = '{client_account}'" for col in num_cols_ok)

# Июнь: любая из дат похожа на 2026-06...
# Если в ODS даты вида 21.06.2026 / 21/06/2026 — раскомментируйте alt_date_where.
date_where = ' OR '.join(f"{col} like '{date_month_like}'" for col in date_cols_ok)
# alt_date_where = ' OR '.join(
#     f"({col} like '%.06.2026%' or {col} like '%/06/2026%' or {col} like '%/06/26%')"
#     for col in date_cols_ok
# )

sql_client = f"""
select
    {select_sql}
from {table_name}
where ({date_where})
  and ({account_where})
"""

print('=== Как ищем счёт ===')
print('1) номер только в c_num_kt / c_num_dt (varchar)')
print('2) c_acc_* не трогаем в WHERE — это decimal id')
print('3) июнь по c_date_prov / c_date_inp / c_date_rec (varchar like 2026-06%)')
print()
print('account_where:', account_where)
print('date_where:', date_where)
print(sql_client)

t0 = time.perf_counter()
with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    gold_hit_df = imp.fetch(sql_client)
print(f'rows={len(gold_hit_df):,}, elapsed={round(time.perf_counter() - t0, 2)}s')

if gold_hit_df is None:
    gold_hit_df = pd.DataFrame()

if not gold_hit_df.empty:
    display(gold_hit_df.head(preview_limit))
    print('\n--- sample c_nazn ---')
    for i, v in enumerate(gold_hit_df['c_nazn'].fillna('').astype(str).unique().tolist()[:50], 1):
        print(f'{i}. {v}')
    mask_ekv = gold_hit_df['c_nazn'].fillna('').astype(str).str.lower().str.contains('эквайр', regex=False)
    ekv_client_df = gold_hit_df.loc[mask_ekv].copy()
    print(f'\nWith эквайр in nazn: {len(ekv_client_df):,}')
    if not ekv_client_df.empty:
        display(ekv_client_df.head(preview_limit))
    gold_hit_df.to_csv(out_client_hit_path, index=False)
    gold_hit_df.to_csv(out_hit_path, index=False)
    print(f'Saved: {out_client_hit_path}')
else:
    print('Счёт не найден.')
    print('Если даты в ODS не ISO (не 2026-06-..), включите alt_date_where в ячейке и перезапустите.')

## 3) Похожие назначения: `зачисление … эквайринг` за тот же период

Массовый список уникальных `c_nazn` того же семейства (только `main_docum`).

In [ ]:
# Опционально: назначения с эквайрингом только среди уже найденных проводок клиента
similar_nazn_df = pd.DataFrame(columns=['c_nazn', 'cnt'])

if 'gold_hit_df' in globals() and not gold_hit_df.empty and nazn_col in gold_hit_df.columns:
    similar_nazn_df = (
        gold_hit_df.assign(_nazn=gold_hit_df[nazn_col].fillna('').astype(str))
        .loc[lambda d: d['_nazn'].str.lower().str.contains('эквайр', regex=False)]
        .groupby('_nazn', as_index=False)
        .size()
        .rename(columns={'_nazn': 'c_nazn', 'size': 'cnt'})
        .sort_values('cnt', ascending=False)
    )
    display(similar_nazn_df.head(preview_limit))
    similar_nazn_df.to_csv(out_similar_nazn_path, index=False)
    print(f'Saved: {out_similar_nazn_path}')
else:
    print('SKIP: нет проводок клиента — похожие nazn не считаем')

## 4) Итог

In [ ]:
summary = pd.DataFrame([
    {'item': 'table', 'value': table_name},
    {'item': 'period', 'value': f'[{date_from}, {date_to_exclusive})'},
    {'item': 'client_account', 'value': client_account},
    {'item': 'account_cols_used', 'value': ','.join(resolved_account_cols) if 'resolved_account_cols' in globals() else ''},
    {'item': 'nazn_col', 'value': nazn_col},
    {'item': 'date_col', 'value': date_col},
    {'item': 'amount_col', 'value': amount_col},
    {'item': 'client_hit_rows', 'value': len(gold_hit_df) if 'gold_hit_df' in globals() else 0},
    {'item': 'client_csv', 'value': out_client_hit_path},
])
display(summary)

if 'gold_hit_df' in globals() and not gold_hit_df.empty:
    print('OK: счёт найден за июнь. Смотрите проводки и c_nazn выше.')
else:
    print('NOT FOUND: счёт не найден за июнь.')